# 11 — Generate reports with Qwen3.5-4B

General multimodal model: raw query image plus retrieved text evidence. Active direction(s): **ECG and CXR**. Generation is append-only and resumable by logical experiment
cell. Exact JSON is preferred, but a usable visible final report is retained and JSON compliance is evaluated
separately. Hidden reasoning channels are not stored.

## Revised generation protocol (v3)
Restart the kernel after uploading the updated Python modules. Existing results are preserved.
The default run is `generation_v3`; outputs are under `experiment4/generations/runs/generation_v3`.
Start with the default **50-cell stratified pilot**, inspect visible reports, failure categories,
and output-limit flags, then set the row limit to `0` for all remaining unattempted cells.
The row limit is per invocation, NOT a cap on total JSONL lines.
One attempt per cell is allowed in this run. A restart skips both successful and failed attempts;
failed cells remain explicitly incomplete. Do not retry deterministic failures without changing the
protocol. Any settings/code/context change requires a new run name, with all comparison cells rerun.
Image resolution and evidence character caps are unchanged. Input budgets include image-expanded
tokens and reserve output capacity. A larger budget may require more GPU memory.
Notebooks 11B/12B/13B are disabled legacy utilities; do not use them for this run.
Use the same `E4_GENERATION_RUN` in 13E and 14-16. Different models can run on separate nodes;
two writers for the same model/run are blocked.

In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get('EXPERIMENT4_CODE_ROOT', str(Path.cwd())))
if not (CODE_ROOT / 'e4_common.py').is_file() and 'EXPERIMENT4_CODE_ROOT' not in os.environ:
    CODE_ROOT = Path.cwd()
if not CODE_ROOT.is_absolute() or not (CODE_ROOT / 'e4_common.py').is_file():
    raise FileNotFoundError('Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 source directory')
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
from e4_common import ARTIFACT_ROOT, EXPERIMENT4_ROOT, SYMILE_ROOT
print({'code_root': str(CODE_ROOT), 'artifact_root': str(ARTIFACT_ROOT),
       'experiment4_root': str(EXPERIMENT4_ROOT), 'symile_root': str(SYMILE_ROOT)})

In [ ]:
# Use the SAME run name in every generation and evaluation notebook.
# A settings/code/context change requires a NEW name and a complete comparison run.
os.environ.setdefault('E4_GENERATION_RUN', 'generation_v3')
from e4_generation_runtime import generation_root
print('Selected generation output:', generation_root(EXPERIMENT4_ROOT))

In [ ]:
import os
from getpass import getpass
from pathlib import Path
from huggingface_hub import HfApi, get_token, login

MODEL_REPO = 'Qwen/Qwen3.5-4B'
token_file_value = os.environ.get('HF_TOKEN_FILE')
HF_TOKEN_FILE = Path(token_file_value).expanduser() if token_file_value else None
hf_token = HF_TOKEN_FILE.read_text().strip() if HF_TOKEN_FILE is not None and HF_TOKEN_FILE.is_file() else None
if hf_token and any(char.isspace() for char in hf_token):
    raise ValueError(f'Hugging Face token file must contain exactly one token: {HF_TOKEN_FILE}')
hf_token = (hf_token or os.environ.get('HF_TOKEN') or
            os.environ.get('HUGGING_FACE_HUB_TOKEN') or get_token())
if not hf_token:
    hf_token = getpass('Enter a Hugging Face read token (input is hidden): ').strip()
    if not hf_token:
        raise RuntimeError('A Hugging Face token is required before loading the model.')
login(token=hf_token, add_to_git_credential=False)
hf_api = HfApi()
try:
    hf_account = hf_api.whoami(token=hf_token)
    hf_model = hf_api.model_info(MODEL_REPO, token=hf_token)
except Exception as exc:
    raise RuntimeError(
        f'Authenticated token cannot access {MODEL_REPO}. Accept any model license/access terms '
        'on Hugging Face, then rerun this cell.') from exc
print({'huggingface_user': hf_account.get('name', 'authenticated'),
       'model_repo': MODEL_REPO, 'revision': hf_model.sha,
       'token_source': str(HF_TOKEN_FILE) if HF_TOKEN_FILE is not None and HF_TOKEN_FILE.is_file() else 'environment/cache/prompt'})

In [ ]:
import os
from e4_generation import run_model
protocol = run_model('qwen35_4b', max_new_tokens=int(os.environ.get('E4_MAX_NEW_TOKENS','1024')),
                     max_input_tokens=int(os.environ.get('E4_MAX_INPUT_TOKENS','12288')),
                     max_rows=int(os.environ.get('E4_MAX_GENERATION_ROWS','50')))
protocol